In [1]:
# ==========================================================
# Cell 1 - Import Libraries
# ==========================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

import rasterio

import matplotlib.pyplot as plt

print("="*60)
print("Notebook 5 Started")
print("="*60)

Notebook 5 Started


In [2]:
# ==========================================================
# Cell 2 - Project Paths
# ==========================================================

PROJECT_DIR = Path("/Users/rajdeepmandal/Desktop/PrithviX")

DATA_DIR = PROJECT_DIR / "Data"
PROCESSED_DIR = PROJECT_DIR / "Processed"

DEM_FILE = DATA_DIR / "DEM" / "ner_dem.tif"

NDVI_FILE = PROCESSED_DIR / "ndvi.tif"

SOIL_FILE = PROCESSED_DIR / "soil.tif"

POP_FILE = DATA_DIR / "Population" / "northeastern_population.tif"

LANDSLIDE_MASK = PROCESSED_DIR / "landslide_mask.tif"

print("="*60)
print("FILES")
print("="*60)

print("DEM        :", DEM_FILE.exists())
print("NDVI       :", NDVI_FILE.exists())
print("SOIL       :", SOIL_FILE.exists())
print("Population :", POP_FILE.exists())
print("Mask       :", LANDSLIDE_MASK.exists())

FILES
DEM        : True
NDVI       : True
SOIL       : True
Population : True
Mask       : True


In [3]:
# ==========================================================
# Cell 3 - Load All Layers
# ==========================================================

with rasterio.open(DEM_FILE) as src:
    dem = src.read(1)

with rasterio.open(NDVI_FILE) as src:
    ndvi = src.read(1)

with rasterio.open(SOIL_FILE) as src:
    soil = src.read(1)

with rasterio.open(POP_FILE) as src:
    population = src.read(1)

with rasterio.open(LANDSLIDE_MASK) as src:
    landslide = src.read(1)

print("="*60)
print("Layers Loaded")
print("="*60)

print("DEM :", dem.shape)
print("NDVI :", ndvi.shape)
print("SOIL :", soil.shape)
print("POP :", population.shape)
print("MASK :", landslide.shape)

Layers Loaded
DEM : (27474, 38963)
NDVI : (27475, 38962)
SOIL : (837, 885)
POP : (8243, 11690)
MASK : (27474, 38963)


In [4]:
# ==========================================================
# Cell 4 - DEM Reference
# ==========================================================

import rasterio

with rasterio.open(DEM_FILE) as src:

    print("="*60)
    print("DEM Reference")
    print("="*60)

    print("Width :", src.width)
    print("Height:", src.height)
    print("CRS   :", src.crs)
    print("Transform:")
    print(src.transform)

DEM Reference
Width : 38963
Height: 27474
CRS   : EPSG:4326
Transform:
| 0.00, 0.00, 88.00|
| 0.00,-0.00, 29.90|
| 0.00, 0.00, 1.00|


In [5]:
print("DEM :", dem.shape)
print("NDVI:", ndvi.shape)
print("SOIL:", soil.shape)
print("POP :", population.shape)
print("MASK:", landslide.shape)

DEM : (27474, 38963)
NDVI: (27475, 38962)
SOIL: (837, 885)
POP : (8243, 11690)
MASK: (27474, 38963)


In [7]:
# ==========================================================
# Cell 5 - Resample All Layers to DEM Grid
# ==========================================================

import numpy as np
from rasterio.warp import reproject, Resampling

# Open DEM again to get profile
with rasterio.open(DEM_FILE) as src:
    dem_profile = src.profile
    dem_transform = src.transform
    dem_crs = src.crs
    dem_height = src.height
    dem_width = src.width

print("="*60)
print("DEM Reference")
print("="*60)
print("Target Shape :", (dem_height, dem_width))

def match_to_dem(input_file, resampling=Resampling.nearest):

    with rasterio.open(input_file) as src:

        destination = np.zeros(
            (dem_height, dem_width),
            dtype=src.dtypes[0]
        )

        reproject(
            source=rasterio.band(src, 1),
            destination=destination,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=dem_transform,
            dst_crs=dem_crs,
            resampling=resampling
        )

    return destination


# Resample layers
ndvi = match_to_dem(NDVI_FILE, Resampling.bilinear)
soil = match_to_dem(SOIL_FILE, Resampling.nearest)
population = match_to_dem(POP_FILE, Resampling.bilinear)
landslide = match_to_dem(LANDSLIDE_MASK, Resampling.nearest)

print("\nResampling Complete")
print("NDVI :", ndvi.shape)
print("SOIL :", soil.shape)
print("POP  :", population.shape)
print("MASK :", landslide.shape)

DEM Reference
Target Shape : (27474, 38963)

Resampling Complete
NDVI : (27474, 38963)
SOIL : (27474, 38963)
POP  : (27474, 38963)
MASK : (27474, 38963)


In [12]:
# ==========================================================
# Cell 5 - Stratified Sampling
# ==========================================================

import numpy as np

print("=" * 60)
print("Creating Balanced Training Dataset")
print("=" * 60)

# Flatten rasters
dem_f = dem.ravel()
ndvi_f = ndvi.ravel()
soil_f = soil.ravel()
pop_f = population.ravel()
mask_f = landslide.ravel()

# Valid pixels
valid = (
    np.isfinite(dem_f) &
    np.isfinite(ndvi_f) &
    np.isfinite(soil_f) &
    np.isfinite(pop_f)
)

# Landslide and non-landslide indices
landslide_idx = np.where((mask_f == 1) & valid)[0]
nonland_idx = np.where((mask_f == 0) & valid)[0]

print("Landslide pixels :", len(landslide_idx))
print("Non-landslide pixels :", len(nonland_idx))

# Sample non-landslide pixels
np.random.seed(42)

NEGATIVE_SAMPLES = 50000

if len(nonland_idx) > NEGATIVE_SAMPLES:
    nonland_sample = np.random.choice(
        nonland_idx,
        NEGATIVE_SAMPLES,
        replace=False
    )
else:
    nonland_sample = nonland_idx

# Combine indices
sample_idx = np.concatenate([landslide_idx, nonland_sample])

# Shuffle
np.random.shuffle(sample_idx)

# Create feature matrix
X = np.column_stack([
    dem_f[sample_idx],
    ndvi_f[sample_idx],
    soil_f[sample_idx],
    pop_f[sample_idx]
]).astype(np.float32)

y = mask_f[sample_idx].astype(np.uint8)

print("\nFeature Matrix :", X.shape)
print("Labels :", y.shape)

print("\nPositive :", np.sum(y == 1))
print("Negative :", np.sum(y == 0))

Creating Balanced Training Dataset
Landslide pixels : 931
Non-landslide pixels : 343518987

Feature Matrix : (50931, 4)
Labels : (50931,)

Positive : 931
Negative : 50000


In [13]:
# ==========================================================
# Cell 6 - Create Training DataFrame
# ==========================================================

import pandas as pd

dataset = pd.DataFrame({
    "DEM": X[:, 0],
    "NDVI": X[:, 1],
    "Soil": X[:, 2],
    "Population": X[:, 3],
    "Landslide": y
})

print("=" * 60)
print("TRAINING DATASET")
print("=" * 60)

print("Shape:", dataset.shape)

display(dataset.head())

print("\nClass Distribution:")
print(dataset["Landslide"].value_counts())

TRAINING DATASET
Shape: (50931, 5)


,DEM,NDVI,Soil,Population,Landslide
0,61.0,0.414891,0.0,8.558760,0
1,86.0,0.879887,0.0,0.856890,0
2,255.0,0.574537,0.0,5.735787,0
3,1895.0,0.876970,0.0,0.014362,0
4,600.0,0.883863,0.0,0.651973,0



Class Distribution:
Landslide
0    50000
1      931
Name: count, dtype: int64


In [14]:
# ==========================================================
# Cell 7 - Save Training Dataset
# ==========================================================

output_csv = PROCESSED_DIR / "training_dataset.csv"

dataset.to_csv(output_csv, index=False)

print("=" * 60)
print("DATASET SAVED")
print("=" * 60)
print(output_csv)

DATASET SAVED
/Users/rajdeepmandal/Desktop/PrithviX/Processed/training_dataset.csv
